# Dueling DQN and REINFORCE Stability & Convergence Analysis

This notebook implements Dueling Deep Q-Networks (Type-1 and Type-2) and REINFORCE (with and without baseline) from scratch. The models are trained on `CartPole-v1` and `Acrobot-v1` using Gymnasium benchmarks.

## Goals
- Implement Type-1 and Type-2 Dueling DQN architectures.
- Implement REINFORCE policy gradient with optional baseline.
- Train on `CartPole-v1` and `Acrobot-v1`.
- Compare stability and convergence across 5 random seeds.
- Evaluate mean and variance of episodic returns.

## Dependencies

Install the required libraries before running this notebook if not already present.

In [2]:
import sys
import subprocess
from importlib import metadata

required = ['gymnasium', 'numpy', 'torch', 'matplotlib', 'tqdm']
installed = {dist.metadata['Name'].lower() for dist in metadata.distributions()}
missing = [pkg for pkg in required if pkg not in installed]
if missing:
    python = sys.executable
    subprocess.check_call([python, '-m', 'pip', 'install', *missing])

print('Dependencies installed or already present:', required)

Dependencies installed or already present: ['gymnasium', 'numpy', 'torch', 'matplotlib', 'tqdm']


## Common Imports and Utilities

Set up shared utilities for seeding, environment creation, evaluation, and plotting.

In [3]:
import gymnasium as gym
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque, namedtuple
from matplotlib import pyplot as plt
from tqdm import trange

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_env(env_name, seed=None):
    env = gym.make(env_name, render_mode=None)
    if seed is not None:
        env.reset(seed=seed)
        env.action_space.seed(seed)
    return env

def plot_results(results, title, xlabel='Episode', ylabel='Return'):
    plt.figure(figsize=(10, 5))
    for label, values in results.items():
        mean_returns = np.mean(values, axis=0)
        std_returns = np.std(values, axis=0)
        x = np.arange(len(mean_returns)) + 1
        plt.plot(x, mean_returns, label=label)
        plt.fill_between(x, mean_returns - std_returns, mean_returns + std_returns, alpha=0.2)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(True)
    plt.show()

## Replay Buffer for DQN

A simple experience replay buffer used by both Dueling DQN variants.

In [ ]:
Transition = namedtuple('Transition', ['state', 'action', 'reward', 'next_state', 'done'])

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[idx] for idx in indices]
        states = torch.tensor(np.array([b.state for b in batch]), dtype=torch.float32, device=device)
        actions = torch.tensor([b.action for b in batch], dtype=torch.int64, device=device).unsqueeze(1)
        rewards = torch.tensor([b.reward for b in batch], dtype=torch.float32, device=device).unsqueeze(1)
        next_states = torch.tensor(np.array([b.next_state for b in batch]), dtype=torch.float32, device=device)
        dones = torch.tensor([b.done for b in batch], dtype=torch.float32, device=device).unsqueeze(1)
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

## Dueling DQN Architectures

Type-1 separates value and advantage streams late in the network. Type-2 uses an additional shared representation in a single head.

In [ ]:
class DuelingDQNType1(nn.Module):
    def __init__(self, obs_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, action_dim)
        )

    def forward(self, x):
        x = self.feature(x)
        value = self.value_stream(x)
        advantage = self.advantage_stream(x)
        return value + advantage - advantage.mean(dim=1, keepdim=True)

class DuelingDQNType2(nn.Module):
    def __init__(self, obs_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU()
        )
        self.value_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        self.advantage_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )

    def forward(self, x):
        x = self.shared(x)
        value = self.value_head(x)
        advantage = self.advantage_head(x)
        return value + advantage - advantage.mean(dim=1, keepdim=True)

## REINFORCE Policy and Baseline

A simple policy network with optional value baseline for variance reduction.

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, obs_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )

    def forward(self, x):
        return F.softmax(self.model(x), dim=-1)

class BaselineNetwork(nn.Module):
    def __init__(self, obs_dim, hidden_dim=128):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze(-1)

## Dueling DQN Training Loop

Training uses epsilon-greedy exploration, Polyak target updates, and experience replay.

In [ ]:
def update_target_network(source, target, tau=1.0):
    for source_param, target_param in zip(source.parameters(), target.parameters()):
        target_param.data.copy_(tau * source_param.data + (1.0 - tau) * target_param.data)

def dqn_train(env_name, model_cls, seed, episodes=200, gamma=0.99, lr=1e-3, buffer_size=10000,
              batch_size=64, min_buffer=500, eps_start=1.0, eps_end=0.05, eps_decay=300, tau=0.005,
              hidden_dim=128):
    set_global_seed(seed)
    env = make_env(env_name, seed)
    obs_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    policy_net = model_cls(obs_dim, action_dim, hidden_dim).to(device)
    target_net = model_cls(obs_dim, action_dim, hidden_dim).to(device)
    target_net.load_state_dict(policy_net.state_dict())
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    replay_buffer = ReplayBuffer(buffer_size)

    returns = []
    state, _ = env.reset(seed=seed)
    for episode in range(episodes):
        episode_return = 0.0
        state, _ = env.reset()
        done = False
        while not done:
            epsilon = eps_end + (eps_start - eps_end) * np.exp(-1.0 * episode / eps_decay)
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    qs = policy_net(torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0))
                    action = int(qs.argmax(dim=1).item())
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            replay_buffer.push(state, action, reward, next_state, float(done))
            state = next_state
            episode_return += reward

            if len(replay_buffer) >= min_buffer:
                states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)
                with torch.no_grad():
                    next_q_values = target_net(next_states).max(dim=1, keepdim=True)[0]
                    targets = rewards + gamma * next_q_values * (1 - dones)
                q_values = policy_net(states).gather(1, actions)
                loss = F.mse_loss(q_values, targets)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                update_target_network(policy_net, target_net, tau)

        returns.append(episode_return)
    env.close()
    return returns

## REINFORCE Training Loop

Includes optional state-value baseline to reduce variance.

In [ ]:
def reinforce_train(env_name, seed, episodes=200, gamma=0.99, lr=1e-3, batch_size=1, use_baseline=False, hidden_dim=128):
    set_global_seed(seed)
    env = make_env(env_name, seed)
    obs_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    policy_net = PolicyNetwork(obs_dim, action_dim, hidden_dim).to(device)
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    baseline_net = BaselineNetwork(obs_dim, hidden_dim).to(device) if use_baseline else None
    baseline_optimizer = optim.Adam(baseline_net.parameters(), lr=lr) if use_baseline else None

    returns = []
    for episode in range(episodes):
        state, _ = env.reset()
        episode_return = 0.0
        rewards = []
        log_probs = []
        states = []
        done = False
        while not done:
            state_tensor = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            probs = policy_net(state_tensor)
            distribution = torch.distributions.Categorical(probs)
            action = distribution.sample().item()
            log_probs.append(distribution.log_prob(torch.tensor(action, device=device)))
            states.append(state_tensor)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            rewards.append(reward)
            state = next_state
            episode_return += reward

        returns.append(episode_return)

        G = 0
        policy_loss = []
        values = []
        if use_baseline:
            baselines = [baseline_net(s).item() for s in states]
        for t in reversed(range(len(rewards))):
            G = rewards[t] + gamma * G
            if use_baseline:
                advantage = G - baselines[t]
            else:
                advantage = G
            policy_loss.append(-log_probs[t] * advantage)
            if use_baseline:
                values.append((torch.tensor([G], dtype=torch.float32, device=device) - baseline_net(states[t])) ** 2)

        optimizer.zero_grad()
        policy_loss = torch.stack(policy_loss).sum()
        policy_loss.backward()
        optimizer.step()

        if use_baseline:
            baseline_optimizer.zero_grad()
            baseline_loss = torch.stack(values).sum()
            baseline_loss.backward()
            baseline_optimizer.step()

    env.close()
    return returns

## Experiment Runner

Execute each algorithm across multiple random seeds and collect summary statistics.

In [ ]:
def run_experiments(env_name, config, seeds=[0, 1, 2, 3, 4]):
    results = {}
    for label, settings in config.items():
        all_returns = []
        print(f'Running {label} on {env_name}')
        for seed in seeds:
            set_global_seed(seed)
            if settings['type'] == 'dqn':
                returns = dqn_train(env_name, settings['model_cls'], seed, **settings['params'])
            elif settings['type'] == 'reinforce':
                returns = reinforce_train(env_name, seed, **settings['params'])
            else:
                raise ValueError('Unknown algorithm type')
            all_returns.append(returns)
        results[label] = np.array(all_returns)
    return results

def summarize_results(results):
    summary = {}
    for label, runs in results.items():
        summary[label] = {
            'mean_final': float(np.mean(runs[:, -1])),
            'std_final': float(np.std(runs[:, -1])),
            'mean_return': float(np.mean(np.mean(runs, axis=1))),
            'std_return': float(np.std(np.mean(runs, axis=1)))
        }
    return summary

## Run Baseline Experiments

We will compare Dueling DQN Type-1, Dueling DQN Type-2, REINFORCE, and REINFORCE with baseline.

In [ ]:
experiment_config = {
    'DuelingDQN-Type1': {
        'type': 'dqn',
        'model_cls': DuelingDQNType1,
        'params': {
            'episodes': 200,
            'gamma': 0.99,
            'lr': 2e-4,
            'buffer_size': 20000,
            'batch_size': 64,
            'min_buffer': 1000,
            'eps_start': 1.0,
            'eps_end': 0.02,
            'eps_decay': 150,
            'tau': 0.01,
            'hidden_dim': 128
        }
    },
    'DuelingDQN-Type2': {
        'type': 'dqn',
        'model_cls': DuelingDQNType2,
        'params': {
            'episodes': 200,
            'gamma': 0.99,
            'lr': 2e-4,
            'buffer_size': 20000,
            'batch_size': 64,
            'min_buffer': 1000,
            'eps_start': 1.0,
            'eps_end': 0.02,
            'eps_decay': 150,
            'tau': 0.01,
            'hidden_dim': 128
        }
    },
    'REINFORCE': {
        'type': 'reinforce',
        'params': {
            'episodes': 200,
            'gamma': 0.99,
            'lr': 1e-3,
            'use_baseline': False,
            'hidden_dim': 128
        }
    },
    'REINFORCE-Baseline': {
        'type': 'reinforce',
        'params': {
            'episodes': 200,
            'gamma': 0.99,
            'lr': 1e-3,
            'use_baseline': True,
            'hidden_dim': 128
        }
    },
}

# Run on CartPole-v1
cartpole_results = run_experiments('CartPole-v1', experiment_config, seeds=[0, 1, 2, 3, 4])
cartpole_summary = summarize_results(cartpole_results)
print('CartPole Summary')
for label, stats in cartpole_summary.items():
    print(f'{label}: mean_final={stats[mean_final]:.2f}, std_final={stats[std_final]:.2f}, mean_return={stats[mean_return]:.2f}, std_return={stats[std_return]:.2f}')
plot_results({k: v for k, v in cartpole_results.items()}, 'CartPole-v1: Mean Return With Std')

## Repeat Experiments on Acrobot-v1

Acrobot is a more challenging control task; the same algorithms are compared to assess stability and convergence under sparser rewards.

In [ ]:
acrobot_config = experiment_config.copy()
acrobot_config['DuelingDQN-Type1']['params']['episodes'] = 300
acrobot_config['DuelingDQN-Type2']['params']['episodes'] = 300
acrobot_config['REINFORCE']['params']['episodes'] = 300
acrobot_config['REINFORCE-Baseline']['params']['episodes'] = 300

acrobot_results = run_experiments('Acrobot-v1', acrobot_config, seeds=[0, 1, 2, 3, 4])
acrobot_summary = summarize_results(acrobot_results)
print('Acrobot Summary')
for label, stats in acrobot_summary.items():
    print(f'{label}: mean_final={stats[mean_final]:.2f}, std_final={stats[std_final]:.2f}, mean_return={stats[mean_return]:.2f}, std_return={stats[std_return]:.2f}')
plot_results({k: v for k, v in acrobot_results.items()}, 'Acrobot-v1: Mean Return With Std')

## Analysis and Discussion

Use the generated plots and summary statistics to compare stability, convergence speed, and regret across the algorithms.

### Suggested analysis points
- Compare final mean returns and standard deviations across seeds.
- Identify whether Dueling DQN Type-1 or Type-2 is more stable.
- Evaluate the benefit of the REINFORCE baseline in variance reduction for episodic returns.
- Observe convergence behavior in CartPole versus Acrobot.
- Discuss how experience replay, Polyak updates, and baselines affect training stability.